In [1]:
import torch

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

# Data Loading and Processing

### Load Data w/ Data Augmentation

In [3]:
from torchvision.transforms import v2
from torchvision.datasets import CIFAR10
from torch import nn

CIFAR_MEAN = (0.49145, 0.48219, 0.44658)
CIFAR_STD = (0.24687, 0.24332, 0.26137)

crop_transform = nn.ModuleList([
    v2.RandomCrop(
        size=32,
        padding=4
    )
])

color_transform = nn.ModuleList([
    v2.ColorJitter(
        brightness=[0.3, 0.7],
        contrast=[0.3, 0.7],
        saturation=[0.3, 0.7]
    )
])

train_transforms = v2.Compose([
    v2.RandomApply(crop_transform, p=0.8),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomApply(color_transform, p=0.75),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD)
])

test_transforms = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD)
])

train_dataset = CIFAR10(
    root="../data",
    train=True,
    transform=train_transforms,
    download=True
)

test_dataset = CIFAR10(
    root="../data",
    train=False,
    transform=test_transforms,
    download=True
)

categories = tuple(train_dataset.classes)

### Plot Images

In [ ]:
from matplotlib import pyplot as plt

fig, axes = plt.subplots(3, 3, figsize=(10, 10))

for i, ax in enumerate(axes.flatten()):
    image, target = train_dataset[i]

    # un-standardize image
    cifar_std = torch.tensor(CIFAR_STD).view(3, 1, 1)
    cifar_mean = torch.tensor(CIFAR_MEAN).view(3, 1, 1)
    image = image * cifar_std
    image = image + cifar_mean
    image = image.permute(1, 2, 0)

    # display images
    ax.imshow(image)
    ax.axis(False)
    ax.set_title(f"Target: {categories[target]}")

plt.show()

In [ ]:
from torch.utils.data import DataLoader, random_split

# create validation set 
generator = torch.Generator().manual_seed(7)
train_dataset, val_dataset = random_split(train_dataset, [0.8, 0.2], generator=generator)

# create DataLoader objects
train_dl = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_dl = DataLoader(val_dataset, batch_size=256, shuffle=False)
test_dl = DataLoader(test_dataset, batch_size=256, shuffle=False)